In [ ]:
# !pip install yfinance
# !pip install TA-Lib 
# !pip install numpy
# !pip install pandas
# !pip install vectorbt
# !pip install scipy

In [ ]:
# Clone repo for lib/ access (Colab only)
import os
if not os.path.exists('/content/trading-strategies'):
    !git clone https://github.com/r-giov/trading-strategies.git /content/trading-strategies
os.chdir('/content/trading-strategies')

In [ ]:
import sys, os
# Ensure lib/ is importable (for Google Sheets logging)
for _p in ['/content/trading-strategies', os.path.join(os.getcwd(), '..')]:
    if os.path.isdir(os.path.join(_p, 'lib')) and _p not in sys.path:
        sys.path.insert(0, _p)

import yfinance as yf
import talib
import numpy as np
import pandas as pd
import vectorbt as vbt
import warnings
from scipy import stats
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')
pd.set_option('future.no_silent_downcasting', True)
from lib.data_manager import download, download_multi, setup_colab_secrets


In [ ]:
# Data: yfinance (daily) | Alpaca (intraday) — auto-selected by data_manager
# For Colab: run setup_colab_secrets() if using Alpaca intraday features
# DOWNLOAD STOCK DATA FROM 2018 USING YFINANCE

# Configuration - Change these variables as needed
TICKER = 'BTC-USD'  # Any ticker symbol (e.g., 'AAPL', 'MSFT', 'GOOGL')
START_DATE = '2018-01-01'  # Any start date in YYYY-MM-DD format

# Download data from start date onwards
stock_data = download(TICKER, START_DATE)

if not stock_data.empty:
    print(f"Successfully downloaded {len(stock_data)} records for {TICKER} from {START_DATE}")
    print(f"Data range: {stock_data.index.min().date()} to {stock_data.index.max().date()}")
    print("\nFirst 5 rows:")
    print(stock_data.head())
else:
    print(f"Failed to download {TICKER} data from yfinance")

# Display the downloaded data
stock_data


In [ ]:
# TECHNICAL ANALYSIS INDICATORS USING TA-LIB

# Make sure stock_data is available from the previous cell
if "stock_data" not in locals():
    raise ValueError("Please run the stock data download cell first")

# Extract OHLCV data (handling multi-level columns from yfinance)
if isinstance(stock_data.columns, pd.MultiIndex):
    close = stock_data[("Close", TICKER)].values
    high = stock_data[("High", TICKER)].values
    low = stock_data[("Low", TICKER)].values
    open_ = stock_data[("Open", TICKER)].values
    volume = stock_data[("Volume", TICKER)].values
else:
    close = stock_data["Close"].values
    high = stock_data["High"].values
    low = stock_data["Low"].values
    open_ = stock_data["Open"].values
    volume = stock_data["Volume"].values

print(f"Calculating technical indicators for {TICKER}...")

# Simple Moving Averages
sma_20 = talib.SMA(close, timeperiod=20)
sma_50 = talib.SMA(close, timeperiod=50)

# Exponential Moving Averages
ema_12 = talib.EMA(close, timeperiod=12)
ema_26 = talib.EMA(close, timeperiod=26)

# MACD
macd, macdsignal, macdhist = talib.MACD(close, fastperiod=12, slowperiod=26, signalperiod=9)

# RSI
rsi = talib.RSI(close, timeperiod=14)

# Stochastic RSI
stochrsi_k, stochrsi_d = talib.STOCHRSI(close, timeperiod=14, fastk_period=3, fastd_period=3, fastd_matype=0)

# VWAP (manual calculation)
typical_price = (high + low + close) / 3
price_volume = typical_price * volume
cumulative_price_volume = np.cumsum(price_volume)
cumulative_volume = np.cumsum(volume)
vwap = cumulative_price_volume / cumulative_volume

# Schaff Trend Cycle
cycle_period = 10
macd_cycle = talib.EMA(macd, timeperiod=cycle_period)
macd_smooth = talib.EMA(macd_cycle, timeperiod=cycle_period)
highest_macd = talib.MAX(macd_smooth, timeperiod=cycle_period)
lowest_macd = talib.MIN(macd_smooth, timeperiod=cycle_period)
stc_k = 100 * ((macd_smooth - lowest_macd) / (highest_macd - lowest_macd))
stc_d = talib.SMA(stc_k, timeperiod=3)

# Create indicators dataframe
indicators_df = pd.DataFrame({
    "Date": stock_data.index,
    "Close": close,
    "SMA_20": sma_20,
    "SMA_50": sma_50,
    "EMA_12": ema_12,
    "EMA_26": ema_26,
    "MACD": macd,
    "MACD_Signal": macdsignal,
    "MACD_Hist": macdhist,
    "RSI": rsi,
    "StochRSI_K": stochrsi_k,
    "StochRSI_D": stochrsi_d,
    "VWAP": vwap,
    "STC_K": stc_k,
    "STC_D": stc_d
})

print("All technical indicators calculated!")
print(f"Data shape: {indicators_df.shape}")
indicators_df.tail(5)

In [ ]:
# PREPARE PRICE SERIES

warnings.filterwarnings("ignore", message="Degrees of freedom <= 0 for slice", category=RuntimeWarning)
warnings.filterwarnings("ignore", message="invalid value encountered in scalar divide", category=RuntimeWarning)

# Expect stock_data and TICKER already exist
def select_close_series(df, ticker):
    if isinstance(df.columns, pd.MultiIndex):
        if ('Close', ticker) in df.columns:
            s = df[('Close', ticker)]
        else:
            cols = [c for c in df.columns if 'Close' in str(c)]
            if not cols:
                raise KeyError("Close not found")
            s = df[cols[0]]
    else:
        s = df['Close']
    return s.astype(float).squeeze()

close = select_close_series(stock_data, TICKER)
close.name = 'price'

# Simple split
TRAIN_RATIO = 0.60 
split_idx = int(len(close) * TRAIN_RATIO)
train_close = close.iloc[:split_idx].copy()
val_close   = close.iloc[split_idx:].copy()

print(f"Data ready: train={train_close.index[0].date()} \u2192 {train_close.index[-1].date()} | val={val_close.index[0].date()} \u2192 {val_close.index[-1].date()}")

RSI MEAN-REVERSION GRID SEARCH - TRAINING SET
----------------------------------------------

This section performs a comprehensive grid search optimization for the **RSI Mean-Reversion Strategy** using only the **training data**.

The goal is to find the optimal RSI Length / Oversold / Overbought combination that maximizes the Sharpe ratio on unseen data.

**Strategy Logic**: Buy when RSI crosses UP through the oversold threshold (mean-reversion entry). Sell when RSI crosses UP through the overbought threshold (profit-taking exit).

---

In [ ]:
# Define Parameter Ranges for RSI Mean-Reversion

# RSI parameters for mean-reversion strategy
rsi_len_range    = list(range(5, 61, 1))          # 56 values
oversold_range   = list(range(15, 46, 1))          # 31 values
overbought_range = list(range(55, 86, 1))          # 31 values

print("RSI Length Periods (lookback window):")
for i, period in enumerate(rsi_len_range, 1):
    print(f"  {i}. {period} periods")

print("Oversold Thresholds (entry trigger):")
for i, val in enumerate(oversold_range, 1):
    print(f"  {i}. {val}")

print("Overbought Thresholds (exit trigger):")
for i, val in enumerate(overbought_range, 1):
    print(f"  {i}. {val}")

# Generate all valid combinations: overbought > oversold + 10 (enforce minimum band gap)
rsi_combinations = []
for rsi_len in rsi_len_range:
    for oversold in oversold_range:
        for overbought in overbought_range:
            if overbought > oversold + 10:
                rsi_combinations.append((rsi_len, oversold, overbought))

print(f"Generated {len(rsi_combinations)} valid RSI Mean-Reversion combinations")
print("Note: Filter applied — overbought > oversold + 10 (minimum band gap)")
print("\n\U0001f4cb First 10 combinations preview:")
for i, (rsi_len, oversold, overbought) in enumerate(rsi_combinations[:10], 1):
    band_gap = overbought - oversold
    print(f"  {i:2d}. RSI_len: {rsi_len:2d} | Oversold: {oversold:2d} | Overbought: {overbought:2d} (gap={band_gap})")
if len(rsi_combinations) > 10:
    print(f"   ... and {len(rsi_combinations) - 10} more combinations")

print("\nReady to test all combinations on training data!")

In [ ]:
# Initialize RSI Mean-Reversion Results Collection System

# Create empty list to store all backtest results
grid_search_results = []

print("RSI Mean-Reversion Results Collection System Initialized")
print(f"   - Will test {len(rsi_combinations)} RSI parameter combinations")
print("   - Results will be stored in 'grid_search_results' list")

# Define what metrics we will collect (All TradingView-style metrics)
metrics_to_collect = [
    # Strategy Parameters
    "rsi_len",
    "oversold", 
    "overbought",
    
    # Return Metrics
    "total_return",
    "annualized_return",
    
    # Risk-Adjusted Return Metrics
    "sharpe_ratio",
    "sortino_ratio",
    "calmar_ratio",
    "omega_ratio",
    "information_ratio",
    "tail_ratio",
    "deflated_sharpe_ratio",
    
    # Risk Metrics
    "max_drawdown",
    "volatility",
    "ulcer_index",
    
    # Trade Performance Metrics
    "win_rate",
    "total_trades",
    "avg_trade_duration",
    "expectancy",
    "profit_factor", 
    "sqn",
    
    # Win/Loss Analysis
    "payoff_ratio",
    "largest_win",
    "largest_loss",
    "avg_win_amount",
    "avg_loss_amount",
    "winning_streak",
    "losing_streak",
    
    # Additional Ratios
    "recovery_factor",
    "gain_to_pain_ratio",
    "serenity_index"
]

print("Metrics to collect for each RSI combination:")
for i, metric in enumerate(metrics_to_collect, 1):
    print(f"  {i}. {metric.replace('_', ' ').title()}")

print("Ready to start the RSI Mean-Reversion grid search!")


In [ ]:
# RSI MEAN-REVERSION GRID SEARCH ON TRAINING DATA

print("INITIATING RSI MEAN-REVERSION GRID SEARCH OPTIMIZATION")
print("=" * 70)
print(f"Testing Strategy: RSI Mean-Reversion (Oversold Entry / Overbought Exit)")
print(f"Training Period: {train_close.index[0].date()} \u2192 {train_close.index[-1].date()}")
print(f"Initial Capital: $100,000")
print(f"Transaction Costs: 0.05% per trade (fees + slippage)")
print(f"Optimization Metric: Sharpe Ratio (risk-adjusted returns)")
print("=" * 70)

total_combinations = len(rsi_combinations)
print(f"Total combinations to test: {total_combinations}")
print(f"Processing sequentially with progress every 1000 combos\n")

grid_search_results = []
successful_tests = 0
failed_tests = 0
skipped_low_trades = 0

train_close_vals = train_close.values.astype(float)
train_idx = train_close.index
years = max((train_idx[-1] - train_idx[0]).days / 365.25, 1e-9)

for combo_num, (rsi_len, oversold, overbought) in enumerate(rsi_combinations, 1):
    try:
        # Compute RSI
        rsi_vals = talib.RSI(train_close_vals, timeperiod=rsi_len)
        rsi_s = pd.Series(rsi_vals, index=train_idx)
        
        # Signal generation — RSI crosses UP through threshold
        entries_raw = (rsi_s.shift(1) <= oversold) & (rsi_s > oversold)
        exits_raw   = (rsi_s.shift(1) <= overbought) & (rsi_s > overbought)
        
        # Shift both by 1 bar for execution delay
        entries_shifted = entries_raw.shift(1)
        entries = pd.Series(np.where(entries_shifted.isna(), False, entries_shifted), index=train_idx, dtype=bool)
        
        exits_shifted = exits_raw.shift(1)
        exits = pd.Series(np.where(exits_shifted.isna(), False, exits_shifted), index=train_idx, dtype=bool)
        
        # Run backtest
        pf = vbt.Portfolio.from_signals(
            close=train_close,
            entries=entries,
            exits=exits,
            init_cash=100_000,
            fees=0.0005,
            slippage=0.0005,
            freq='D'
        )
        
        trades = pf.trades
        total_trades = len(trades)
        
        # Skip combos with < 10 trades
        if total_trades < 10:
            skipped_low_trades += 1
            continue
        
        trades_per_year = total_trades / years
        
        # Core metrics
        total_return = float(pf.total_return())
        annualized_return = float(pf.annualized_return(freq='D'))
        max_drawdown = float(pf.max_drawdown())
        volatility = float(pf.annualized_volatility(freq='D'))
        sharpe_ratio = float(pf.sharpe_ratio(freq='D'))
        sortino_ratio = float(pf.sortino_ratio(freq='D'))
        
        # Calmar ratio
        calmar_ratio = annualized_return / abs(max_drawdown) if abs(max_drawdown) > 1e-9 else np.nan
        
        # Trade statistics
        win_rate_pct = np.nan
        profit_factor = np.nan
        expectancy = 0.0
        avg_win_amount = 0.0
        avg_loss_amount = 0.0
        largest_win = 0.0
        largest_loss = 0.0
        winning_streak = np.nan
        losing_streak = np.nan
        avg_trade_duration = np.nan
        sqn = np.nan
        
        tr = trades.returns.values if hasattr(trades.returns, 'values') else np.array(trades.returns)
        if tr.size > 0:
            pos = tr[tr > 0]
            neg = tr[tr < 0]
            win_rate_pct = (len(pos) / len(tr)) * 100.0 if len(tr) > 0 else np.nan
            gains = pos.sum() if len(pos) else 0.0
            losses = abs(neg.sum()) if len(neg) else 0.0
            profit_factor = gains / losses if losses > 0 else np.inf
            expectancy = float(tr.mean())
            avg_win_amount = float(pos.mean()) if len(pos) else 0.0
            avg_loss_amount = float(abs(neg.mean())) if len(neg) else 0.0
            largest_win = float(pos.max()) if len(pos) else 0.0
            largest_loss = float(neg.min()) if len(neg) else 0.0
            sqn = float(tr.mean() / tr.std() * np.sqrt(len(tr))) if tr.std() > 0 else np.nan
            
            try:
                winning_streak = int(trades.winning_streak())
                losing_streak = int(trades.losing_streak())
            except:
                pass
        
        # Ulcer Index
        returns = pf.returns()
        cum = (1 + returns).cumprod()
        peak = cum.cummax()
        dd = (cum - peak) / peak
        ulcer_index = float(np.sqrt((dd.pow(2)).mean())) if len(dd) > 0 else np.nan
        
        payoff_ratio = (avg_win_amount / avg_loss_amount) if avg_loss_amount not in (0.0, np.nan) and avg_loss_amount > 0 else np.inf
        
        # Omega ratio (threshold = 0)
        rets = returns.values
        omega_ratio = float(rets[rets > 0].sum() / abs(rets[rets < 0].sum())) if abs(rets[rets < 0].sum()) > 0 else np.inf
        
        # Recovery factor
        recovery_factor = total_return / abs(max_drawdown) if abs(max_drawdown) > 1e-9 else np.nan
        
        # Gain to pain ratio
        gain_to_pain = float(rets.sum() / abs(rets[rets < 0].sum())) if abs(rets[rets < 0].sum()) > 0 else np.inf
        
        # Serenity index = recovery_factor / ulcer_index
        serenity_index = recovery_factor / ulcer_index if ulcer_index and ulcer_index > 0 else np.nan
        
        grid_search_results.append({
            "rsi_len": rsi_len,
            "oversold": oversold,
            "overbought": overbought,
            "total_return": total_return,
            "annualized_return": annualized_return,
            "sharpe_ratio": sharpe_ratio,
            "sortino_ratio": sortino_ratio,
            "calmar_ratio": calmar_ratio,
            "omega_ratio": omega_ratio,
            "information_ratio": np.nan,
            "tail_ratio": np.nan,
            "deflated_sharpe_ratio": np.nan,
            "max_drawdown": max_drawdown,
            "volatility": volatility,
            "ulcer_index": ulcer_index,
            "win_rate": win_rate_pct,
            "total_trades": total_trades,
            "avg_trade_duration": avg_trade_duration,
            "expectancy": expectancy,
            "profit_factor": profit_factor,
            "sqn": sqn,
            "payoff_ratio": payoff_ratio,
            "largest_win": largest_win,
            "largest_loss": largest_loss,
            "avg_win_amount": avg_win_amount,
            "avg_loss_amount": avg_loss_amount,
            "winning_streak": winning_streak,
            "losing_streak": losing_streak,
            "recovery_factor": recovery_factor,
            "gain_to_pain_ratio": gain_to_pain,
            "serenity_index": serenity_index,
            "trades_per_year": trades_per_year
        })
        successful_tests += 1
        
    except Exception as e:
        failed_tests += 1
    
    # Progress every 1000 combos
    if combo_num % 1000 == 0:
        progress_pct = (combo_num / total_combinations) * 100
        print(f"\U0001f504 Progress: {combo_num}/{total_combinations} ({progress_pct:.1f}%) | \u2714 {successful_tests} successful | Skipped (low trades): {skipped_low_trades}")

# SUMMARY
print("\n" + "=" * 70)
print("RSI MEAN-REVERSION GRID SEARCH COMPLETED!")
print("=" * 70)
print(f"Total combinations attempted: {total_combinations}")
print(f"Successfully completed: {successful_tests}")
print(f"Skipped (< 10 trades): {skipped_low_trades}")
print(f"Failed: {failed_tests}")
print(f"Success rate: {(successful_tests/total_combinations)*100:.1f}%")
print(f"\n\u2714 Results stored in 'grid_search_results' ({len(grid_search_results)} entries)")

if successful_tests > 0:
    results_df_preview = pd.DataFrame(grid_search_results)
    
    # Display top 10 combinations
    print("\n" + "=" * 70)
    print("\U0001f3c6 TOP 10 COMBINATIONS (by In-Sample Sharpe Ratio)")
    print("=" * 70)
    
    top_10 = results_df_preview.nlargest(10, 'sharpe_ratio')
    for rank, (idx, row) in enumerate(top_10.iterrows(), 1):
        print(f"\n#{rank} - RSI(len={int(row['rsi_len'])}, OS={int(row['oversold'])}, OB={int(row['overbought'])})")
        print(f"   Sharpe Ratio:      {row['sharpe_ratio']:.3f}")
        print(f"   Total Return:      {row['total_return']:.2%}")
        print(f"   Annualized Return: {row['annualized_return']:.2%}")
        print(f"   Max Drawdown:      {row['max_drawdown']:.2%}")
        print(f"   Win Rate:          {row['win_rate']:.1f}%")
        print(f"   Profit Factor:     {row['profit_factor']:.2f}")
        print(f"   Total Trades:      {int(row['total_trades'])} ({row['trades_per_year']:.1f}/year)")
    
    print("\n" + "=" * 70)



In [ ]:
# TOP 5 OUT-OF-SAMPLE VALIDATION & COMPARISON TABLE

if 'FREQ' not in globals():
    FREQ = "1D"

results_df = pd.DataFrame(grid_search_results)

if results_df.empty:
    print("No results to validate.")
else:
    print("=" * 90)
    print("\U0001f3c6 TOP 5 STRATEGIES - OUT-OF-SAMPLE VALIDATION")
    print("=" * 90)
    print(f"Training Period: {train_close.index[0].date()} \u2192 {train_close.index[-1].date()}")
    print(f"Validation Period: {val_close.index[0].date()} \u2192 {val_close.index[-1].date()}")
    print("=" * 90)
    
    # Get top 5 by in-sample Sharpe
    top_5_strategies = results_df.nlargest(5, 'sharpe_ratio')
    
    # Results storage
    oos_results = []
    
    val_close_vals = val_close.values.astype(float)
    val_idx = val_close.index
    
    for idx_row, strategy in top_5_strategies.iterrows():
        rsi_len = int(strategy['rsi_len'])
        oversold = int(strategy['oversold'])
        overbought = int(strategy['overbought'])
        
        # Compute RSI on validation data
        rsi_vals = talib.RSI(val_close_vals, timeperiod=rsi_len)
        rsi_s = pd.Series(rsi_vals, index=val_idx)
        
        # Signal generation
        entries_raw = (rsi_s.shift(1) <= oversold) & (rsi_s > oversold)
        exits_raw   = (rsi_s.shift(1) <= overbought) & (rsi_s > overbought)
        
        # Shift by 1 bar
        entries_shifted = entries_raw.shift(1)
        entries = pd.Series(np.where(entries_shifted.isna(), False, entries_shifted), index=val_idx, dtype=bool)
        exits_shifted = exits_raw.shift(1)
        exits = pd.Series(np.where(exits_shifted.isna(), False, exits_shifted), index=val_idx, dtype=bool)
        
        # Run OOS backtest
        pf_val = vbt.Portfolio.from_signals(
            close=val_close,
            entries=entries,
            exits=exits,
            init_cash=100_000,
            fees=0.0005,
            slippage=0.0005,
            freq=FREQ
        )
        
        # OOS metrics
        oos_total_return = float(pf_val.total_return())
        oos_annualized_return = float(pf_val.annualized_return(freq=FREQ))
        oos_sharpe = float(pf_val.sharpe_ratio(freq=FREQ))
        oos_sortino = float(pf_val.sortino_ratio(freq=FREQ))
        oos_max_drawdown = float(pf_val.max_drawdown())
        oos_volatility = float(pf_val.annualized_volatility(freq=FREQ))
        
        trades = pf_val.trades
        oos_total_trades = len(trades)
        years = max((val_close.index[-1] - val_close.index[0]).days / 365.25, 1e-9)
        oos_trades_per_year = oos_total_trades / years
        
        oos_win_rate_pct = np.nan
        oos_profit_factor = np.nan
        oos_expectancy = 0.0
        
        if oos_total_trades > 0:
            tr = trades.returns.values if hasattr(trades.returns, 'values') else np.array(trades.returns)
            if tr.size > 0:
                pos = tr[tr > 0]
                neg = tr[tr < 0]
                oos_win_rate_pct = (len(pos) / len(tr)) * 100.0 if len(tr) > 0 else np.nan
                gains = pos.sum() if len(pos) else 0.0
                losses = abs(neg.sum()) if len(neg) else 0.0
                oos_profit_factor = gains / losses if losses > 0 else np.inf
                oos_expectancy = float(tr.mean())
        
        # Store results
        oos_results.append({
            'Rank': len(oos_results) + 1,
            'RSI_Len': rsi_len,
            'Oversold': oversold,
            'Overbought': overbought,
            'IS_Sharpe': float(strategy['sharpe_ratio']),
            'IS_Return': float(strategy['total_return']),
            'IS_MaxDD': float(strategy['max_drawdown']),
            'IS_WinRate': float(strategy['win_rate']),
            'OOS_Sharpe': oos_sharpe,
            'OOS_Return': oos_total_return,
            'OOS_MaxDD': oos_max_drawdown,
            'OOS_WinRate': oos_win_rate_pct,
            'OOS_Trades': oos_total_trades,
            'OOS_ProfitFactor': oos_profit_factor,
            'Sharpe_Degradation': ((oos_sharpe - strategy['sharpe_ratio']) / abs(strategy['sharpe_ratio']) * 100) if strategy['sharpe_ratio'] != 0 else np.nan,
            'Return_Degradation': ((oos_total_return - strategy['total_return']) / abs(strategy['total_return']) * 100) if strategy['total_return'] != 0 else np.nan
        })
    
    # Create DataFrame
    oos_df = pd.DataFrame(oos_results)
    
    # Sort by OOS Sharpe (best to worst)
    oos_df_sorted = oos_df.sort_values('OOS_Sharpe', ascending=False).reset_index(drop=True)
    oos_df_sorted['OOS_Rank'] = range(1, len(oos_df_sorted) + 1)
    
    # Display comprehensive table
    print("\n\U0001f4ca COMPREHENSIVE COMPARISON TABLE (Sorted by OOS Sharpe)")
    print("=" * 90)
    
    display_df = pd.DataFrame({
        'IS\u2192OOS Rank': oos_df_sorted['Rank'].astype(str) + '\u2192' + oos_df_sorted['OOS_Rank'].astype(str),
        'Strategy': oos_df_sorted.apply(lambda x: f"RSI({int(x['RSI_Len'])},{int(x['Oversold'])},{int(x['Overbought'])})", axis=1),
        'IS Sharpe': oos_df_sorted['IS_Sharpe'].map('{:.3f}'.format),
        'OOS Sharpe': oos_df_sorted['OOS_Sharpe'].map('{:.3f}'.format),
        'Sharpe \u0394%': oos_df_sorted['Sharpe_Degradation'].map('{:+.1f}%'.format),
        'IS Return': oos_df_sorted['IS_Return'].map('{:.1%}'.format),
        'OOS Return': oos_df_sorted['OOS_Return'].map('{:.1%}'.format),
        'Return \u0394%': oos_df_sorted['Return_Degradation'].map('{:+.1f}%'.format),
        'OOS Trades': oos_df_sorted['OOS_Trades'].astype(int),
        'OOS WinRate': oos_df_sorted['OOS_WinRate'].map('{:.1f}%'.format),
        'OOS PF': oos_df_sorted['OOS_ProfitFactor'].map('{:.2f}'.format)
    })
    
    print(display_df.to_string(index=False))
    
    # Highlight best OOS performer
    best_oos = oos_df_sorted.iloc[0]
    print("\n" + "=" * 90)
    print(f"\u2728 BEST OUT-OF-SAMPLE PERFORMER")
    print("=" * 90)
    print(f"Strategy: RSI(len={int(best_oos['RSI_Len'])}, OS={int(best_oos['Oversold'])}, OB={int(best_oos['Overbought'])})")
    print(f"In-Sample Rank:        #{int(best_oos['Rank'])}")
    print(f"Out-of-Sample Rank:    #1")
    print(f"OOS Sharpe Ratio:      {best_oos['OOS_Sharpe']:.3f}")
    print(f"OOS Return:            {best_oos['OOS_Return']:.2%}")
    print(f"OOS Max Drawdown:      {best_oos['OOS_MaxDD']:.2%}")
    print(f"OOS Win Rate:          {best_oos['OOS_WinRate']:.1f}%")
    print(f"OOS Profit Factor:     {best_oos['OOS_ProfitFactor']:.2f}")
    print(f"OOS Total Trades:      {int(best_oos['OOS_Trades'])}")
    print(f"Sharpe Degradation:    {best_oos['Sharpe_Degradation']:+.1f}%")
    print("=" * 90)
    
    # ── Equity Curve Plots ──
    # Use the best IS strategy for equity curve visualization
    best_is = results_df.loc[results_df['sharpe_ratio'].idxmax()]
    b_rsi_len = int(best_is['rsi_len'])
    b_oversold = int(best_is['oversold'])
    b_overbought = int(best_is['overbought'])
    
    # IS equity curve
    rsi_train = pd.Series(talib.RSI(train_close.values.astype(float), timeperiod=b_rsi_len), index=train_close.index)
    e_raw_tr = (rsi_train.shift(1) <= b_oversold) & (rsi_train > b_oversold)
    x_raw_tr = (rsi_train.shift(1) <= b_overbought) & (rsi_train > b_overbought)
    e_tr = pd.Series(np.where(e_raw_tr.shift(1).isna(), False, e_raw_tr.shift(1)), index=train_close.index, dtype=bool)
    x_tr = pd.Series(np.where(x_raw_tr.shift(1).isna(), False, x_raw_tr.shift(1)), index=train_close.index, dtype=bool)
    pf_train = vbt.Portfolio.from_signals(close=train_close, entries=e_tr, exits=x_tr,
                                           init_cash=100_000, fees=0.0005, slippage=0.0005, freq='D')
    
    # OOS equity curve
    rsi_val = pd.Series(talib.RSI(val_close.values.astype(float), timeperiod=b_rsi_len), index=val_close.index)
    e_raw_vl = (rsi_val.shift(1) <= b_oversold) & (rsi_val > b_oversold)
    x_raw_vl = (rsi_val.shift(1) <= b_overbought) & (rsi_val > b_overbought)
    e_vl = pd.Series(np.where(e_raw_vl.shift(1).isna(), False, e_raw_vl.shift(1)), index=val_close.index, dtype=bool)
    x_vl = pd.Series(np.where(x_raw_vl.shift(1).isna(), False, x_raw_vl.shift(1)), index=val_close.index, dtype=bool)
    pf_val2 = vbt.Portfolio.from_signals(close=val_close, entries=e_vl, exits=x_vl,
                                          init_cash=100_000, fees=0.0005, slippage=0.0005, freq='D')
    
    eq_train = (1 + pf_train.returns()).cumprod()
    eq_val = (1 + pf_val2.returns()).cumprod()
    
    fig, ax = plt.subplots(figsize=(14, 7))
    ax.plot(train_close.index, eq_train.values, label='Train (In-Sample)', color='blue', linewidth=1.5, alpha=0.8)
    ax.plot(val_close.index, eq_val.values, label='Validation (Out-of-Sample)', color='orange', linewidth=1.5, alpha=0.8)
    ax.axvline(x=train_close.index[-1], color='red', linestyle='--', alpha=0.5, label='Train/Val Split')
    ax.set_title(f'RSI(len={b_rsi_len}, OS={b_oversold}, OB={b_overbought}) - Equity Curves', fontsize=14, fontweight='bold')
    ax.set_xlabel('Date', fontsize=12)
    ax.set_ylabel('Cumulative Returns (normalized to 1)', fontsize=12)
    ax.grid(True, alpha=0.3)
    ax.legend(loc='best')
    plt.tight_layout()
    plt.show()



PARAMETER SENSITIVITY ANALYSIS
------------------------------

This section evaluates how sensitive the strategy's performance is to small changes in each parameter.

For each parameter (`rsi_len`, `oversold`, `overbought`), we vary it ±15 around the best value while holding the other two fixed.

**Color coding**: Dark green (>+10% vs base), Light green (0-10%), Orange (-10-0%), Red (<-10%).

---

In [ ]:
# COMPACT PARAMETER SENSITIVITY ANALYSIS - BAR CHARTS

if results_df.empty:
    print("No results available for sensitivity analysis.")
else:
    # Use BEST IS strategy
    best = results_df.loc[results_df['sharpe_ratio'].idxmax()]
    rsi_len_base = int(best['rsi_len'])
    oversold_base = int(best['oversold'])
    overbought_base = int(best['overbought'])
    base_is_sharpe = float(best['sharpe_ratio'])
    base_oos_sharpe = np.nan
    
    # Try to get OOS sharpe from validation results
    try:
        rsi_val_check = pd.Series(talib.RSI(val_close.values.astype(float), timeperiod=rsi_len_base), index=val_close.index)
        e_raw_check = (rsi_val_check.shift(1) <= oversold_base) & (rsi_val_check > oversold_base)
        x_raw_check = (rsi_val_check.shift(1) <= overbought_base) & (rsi_val_check > overbought_base)
        e_check = pd.Series(np.where(e_raw_check.shift(1).isna(), False, e_raw_check.shift(1)), index=val_close.index, dtype=bool)
        x_check = pd.Series(np.where(x_raw_check.shift(1).isna(), False, x_raw_check.shift(1)), index=val_close.index, dtype=bool)
        pf_check = vbt.Portfolio.from_signals(close=val_close, entries=e_check, exits=x_check,
                                              init_cash=100_000, fees=0.0005, slippage=0.0005, freq='D')
        base_oos_sharpe = float(pf_check.sharpe_ratio(freq='D'))
    except:
        pass
    
    print(f"\U0001f52c PARAMETER SENSITIVITY ANALYSIS - Base: RSI(len={rsi_len_base}, OS={oversold_base}, OB={overbought_base})")
    print(f"IS Sharpe: {base_is_sharpe:.3f}" + (f" | OOS Sharpe: {base_oos_sharpe:.3f}" if not np.isnan(base_oos_sharpe) else ""))

    # Create sensitivity ranges (±15 around each parameter)
    rsi_len_sens = sorted(list(set(range(max(2, rsi_len_base - 15), rsi_len_base + 16))))
    oversold_sens = sorted(list(set(range(max(1, oversold_base - 15), oversold_base + 16))))
    overbought_sens = sorted(list(set(range(max(oversold_base + 11, overbought_base - 15), overbought_base + 16))))
    
    combos_rsi_len = [(rl, oversold_base, overbought_base) for rl in rsi_len_sens]
    combos_oversold = [(rsi_len_base, os, overbought_base) for os in oversold_sens]
    combos_overbought = [(rsi_len_base, oversold_base, ob) for ob in overbought_sens if ob > oversold_base + 10]
    all_combos = combos_rsi_len + combos_oversold + combos_overbought

    def eval_combo_both(rl: int, os: int, ob: int) -> dict:
        """Evaluate on both in-sample and out-of-sample"""
        # IN-SAMPLE
        rsi_is = pd.Series(talib.RSI(train_close.values.astype(float), timeperiod=rl), index=train_close.index)
        e_raw_is = (rsi_is.shift(1) <= os) & (rsi_is > os)
        x_raw_is = (rsi_is.shift(1) <= ob) & (rsi_is > ob)
        e_is = pd.Series(np.where(e_raw_is.shift(1).isna(), False, e_raw_is.shift(1)), index=train_close.index, dtype=bool)
        x_is = pd.Series(np.where(x_raw_is.shift(1).isna(), False, x_raw_is.shift(1)), index=train_close.index, dtype=bool)
        pf_is = vbt.Portfolio.from_signals(close=train_close, entries=e_is, exits=x_is,
                                           init_cash=100_000, fees=0.0005, slippage=0.0005, freq='D')
        
        # OUT-OF-SAMPLE
        rsi_oos = pd.Series(talib.RSI(val_close.values.astype(float), timeperiod=rl), index=val_close.index)
        e_raw_oos = (rsi_oos.shift(1) <= os) & (rsi_oos > os)
        x_raw_oos = (rsi_oos.shift(1) <= ob) & (rsi_oos > ob)
        e_oos = pd.Series(np.where(e_raw_oos.shift(1).isna(), False, e_raw_oos.shift(1)), index=val_close.index, dtype=bool)
        x_oos = pd.Series(np.where(x_raw_oos.shift(1).isna(), False, x_raw_oos.shift(1)), index=val_close.index, dtype=bool)
        pf_oos = vbt.Portfolio.from_signals(close=val_close, entries=e_oos, exits=x_oos,
                                            init_cash=100_000, fees=0.0005, slippage=0.0005, freq='D')
        
        return {
            'rsi_len': rl, 'oversold': os, 'overbought': ob,
            'is_sharpe': float(pf_is.sharpe_ratio(freq='D')),
            'is_return': float(pf_is.total_return()),
            'is_maxdd': float(pf_is.max_drawdown()),
            'oos_sharpe': float(pf_oos.sharpe_ratio(freq='D')),
            'oos_return': float(pf_oos.total_return()),
            'oos_maxdd': float(pf_oos.max_drawdown()),
            'oos_trades': len(pf_oos.trades)
        }

    print(f"\n\U0001f504 Testing {len(all_combos)} parameter variations...")
    
    rows = []
    for combo in all_combos:
        try:
            rows.append(eval_combo_both(*combo))
        except Exception:
            pass

    if not rows:
        print("\u274c No sensitivity results computed.")
    else:
        sens_df = pd.DataFrame(rows)
        
        # Split by parameter variation type
        rsi_len_variations = sens_df[(sens_df['oversold'] == oversold_base) & (sens_df['overbought'] == overbought_base)].copy().sort_values('rsi_len')
        oversold_variations = sens_df[(sens_df['rsi_len'] == rsi_len_base) & (sens_df['overbought'] == overbought_base)].copy().sort_values('oversold')
        overbought_variations = sens_df[(sens_df['rsi_len'] == rsi_len_base) & (sens_df['oversold'] == oversold_base)].copy().sort_values('overbought')
        
        # Calculate degradations
        rsi_len_variations['is_sharpe_delta'] = ((rsi_len_variations['is_sharpe'] - base_is_sharpe) / abs(base_is_sharpe) * 100)
        oversold_variations['is_sharpe_delta'] = ((oversold_variations['is_sharpe'] - base_is_sharpe) / abs(base_is_sharpe) * 100)
        overbought_variations['is_sharpe_delta'] = ((overbought_variations['is_sharpe'] - base_is_sharpe) / abs(base_is_sharpe) * 100)
        
        if not np.isnan(base_oos_sharpe):
            rsi_len_variations['oos_sharpe_delta'] = ((rsi_len_variations['oos_sharpe'] - base_oos_sharpe) / abs(base_oos_sharpe) * 100)
            oversold_variations['oos_sharpe_delta'] = ((oversold_variations['oos_sharpe'] - base_oos_sharpe) / abs(base_oos_sharpe) * 100)
            overbought_variations['oos_sharpe_delta'] = ((overbought_variations['oos_sharpe'] - base_oos_sharpe) / abs(base_oos_sharpe) * 100)
        
        # CREATE BAR CHARTS - 2x3 grid
        fig, axes = plt.subplots(2, 3, figsize=(20, 10))
        fig.suptitle(f'Parameter Sensitivity Analysis - Base: RSI(len={rsi_len_base}, OS={oversold_base}, OB={overbought_base})', 
                     fontsize=16, fontweight='bold')
        
        # IN-SAMPLE BAR CHARTS
        # RSI Length IS
        colors1_is = ['red' if x < -10 else 'orange' if x < 0 else 'lightgreen' if x < 10 else 'darkgreen' 
                      for x in rsi_len_variations['is_sharpe_delta']]
        axes[0, 0].bar(rsi_len_variations['rsi_len'], rsi_len_variations['is_sharpe_delta'], color=colors1_is, edgecolor='black', linewidth=0.5)
        axes[0, 0].axhline(0, color='black', linewidth=1.5)
        axes[0, 0].axvline(rsi_len_base, color='blue', linestyle='--', linewidth=2.5, alpha=0.7, label=f'Base={rsi_len_base}')
        axes[0, 0].set_xlabel('RSI Length', fontsize=11, fontweight='bold')
        axes[0, 0].set_ylabel('Sharpe \u0394%', fontsize=11, fontweight='bold')
        axes[0, 0].set_title('RSI Length - In-Sample', fontsize=12, fontweight='bold')
        axes[0, 0].grid(axis='y', alpha=0.3)
        axes[0, 0].legend(fontsize=10)
        
        # Oversold IS
        colors2_is = ['red' if x < -10 else 'orange' if x < 0 else 'lightgreen' if x < 10 else 'darkgreen' 
                      for x in oversold_variations['is_sharpe_delta']]
        axes[0, 1].bar(oversold_variations['oversold'], oversold_variations['is_sharpe_delta'], color=colors2_is, edgecolor='black', linewidth=0.5)
        axes[0, 1].axhline(0, color='black', linewidth=1.5)
        axes[0, 1].axvline(oversold_base, color='blue', linestyle='--', linewidth=2.5, alpha=0.7, label=f'Base={oversold_base}')
        axes[0, 1].set_xlabel('Oversold Threshold', fontsize=11, fontweight='bold')
        axes[0, 1].set_ylabel('Sharpe \u0394%', fontsize=11, fontweight='bold')
        axes[0, 1].set_title('Oversold - In-Sample', fontsize=12, fontweight='bold')
        axes[0, 1].grid(axis='y', alpha=0.3)
        axes[0, 1].legend(fontsize=10)
        
        # Overbought IS
        colors3_is = ['red' if x < -10 else 'orange' if x < 0 else 'lightgreen' if x < 10 else 'darkgreen' 
                      for x in overbought_variations['is_sharpe_delta']]
        axes[0, 2].bar(overbought_variations['overbought'], overbought_variations['is_sharpe_delta'], color=colors3_is, edgecolor='black', linewidth=0.5)
        axes[0, 2].axhline(0, color='black', linewidth=1.5)
        axes[0, 2].axvline(overbought_base, color='blue', linestyle='--', linewidth=2.5, alpha=0.7, label=f'Base={overbought_base}')
        axes[0, 2].set_xlabel('Overbought Threshold', fontsize=11, fontweight='bold')
        axes[0, 2].set_ylabel('Sharpe \u0394%', fontsize=11, fontweight='bold')
        axes[0, 2].set_title('Overbought - In-Sample', fontsize=12, fontweight='bold')
        axes[0, 2].grid(axis='y', alpha=0.3)
        axes[0, 2].legend(fontsize=10)
        
        # OUT-OF-SAMPLE BAR CHARTS
        if not np.isnan(base_oos_sharpe):
            # RSI Length OOS
            colors1_oos = ['red' if x < -10 else 'orange' if x < 0 else 'lightgreen' if x < 10 else 'darkgreen' 
                          for x in rsi_len_variations['oos_sharpe_delta']]
            axes[1, 0].bar(rsi_len_variations['rsi_len'], rsi_len_variations['oos_sharpe_delta'], color=colors1_oos, edgecolor='black', linewidth=0.5)
            axes[1, 0].axhline(0, color='black', linewidth=1.5)
            axes[1, 0].axvline(rsi_len_base, color='blue', linestyle='--', linewidth=2.5, alpha=0.7, label=f'Base={rsi_len_base}')
            axes[1, 0].set_xlabel('RSI Length', fontsize=11, fontweight='bold')
            axes[1, 0].set_ylabel('Sharpe \u0394%', fontsize=11, fontweight='bold')
            axes[1, 0].set_title('RSI Length - Out-of-Sample', fontsize=12, fontweight='bold')
            axes[1, 0].grid(axis='y', alpha=0.3)
            axes[1, 0].legend(fontsize=10)
            
            # Oversold OOS
            colors2_oos = ['red' if x < -10 else 'orange' if x < 0 else 'lightgreen' if x < 10 else 'darkgreen' 
                          for x in oversold_variations['oos_sharpe_delta']]
            axes[1, 1].bar(oversold_variations['oversold'], oversold_variations['oos_sharpe_delta'], color=colors2_oos, edgecolor='black', linewidth=0.5)
            axes[1, 1].axhline(0, color='black', linewidth=1.5)
            axes[1, 1].axvline(oversold_base, color='blue', linestyle='--', linewidth=2.5, alpha=0.7, label=f'Base={oversold_base}')
            axes[1, 1].set_xlabel('Oversold Threshold', fontsize=11, fontweight='bold')
            axes[1, 1].set_ylabel('Sharpe \u0394%', fontsize=11, fontweight='bold')
            axes[1, 1].set_title('Oversold - Out-of-Sample', fontsize=12, fontweight='bold')
            axes[1, 1].grid(axis='y', alpha=0.3)
            axes[1, 1].legend(fontsize=10)
            
            # Overbought OOS
            colors3_oos = ['red' if x < -10 else 'orange' if x < 0 else 'lightgreen' if x < 10 else 'darkgreen' 
                          for x in overbought_variations['oos_sharpe_delta']]
            axes[1, 2].bar(overbought_variations['overbought'], overbought_variations['oos_sharpe_delta'], color=colors3_oos, edgecolor='black', linewidth=0.5)
            axes[1, 2].axhline(0, color='black', linewidth=1.5)
            axes[1, 2].axvline(overbought_base, color='blue', linestyle='--', linewidth=2.5, alpha=0.7, label=f'Base={overbought_base}')
            axes[1, 2].set_xlabel('Overbought Threshold', fontsize=11, fontweight='bold')
            axes[1, 2].set_ylabel('Sharpe \u0394%', fontsize=11, fontweight='bold')
            axes[1, 2].set_title('Overbought - Out-of-Sample', fontsize=12, fontweight='bold')
            axes[1, 2].grid(axis='y', alpha=0.3)
            axes[1, 2].legend(fontsize=10)
        
        plt.tight_layout()
        plt.show()
        
        # COMPACT SUMMARY TABLE
        print("\n\U0001f4cb SENSITIVITY SUMMARY")
        print("=" * 80)
        
        summary_data = []
        for param_name, variations, param_col in [('RSI_Len', rsi_len_variations, 'rsi_len'), 
                                                   ('Oversold', oversold_variations, 'oversold'), 
                                                   ('Overbought', overbought_variations, 'overbought')]:
            summary_data.append({
                'Parameter': param_name,
                'IS Range': f"{variations['is_sharpe'].min():.3f} - {variations['is_sharpe'].max():.3f}",
                'IS Max \u0394%': f"{variations['is_sharpe_delta'].min():.1f}%",
                'OOS Range': f"{variations['oos_sharpe'].min():.3f} - {variations['oos_sharpe'].max():.3f}" if not np.isnan(base_oos_sharpe) else 'N/A',
                'OOS Max \u0394%': f"{variations['oos_sharpe_delta'].min():.1f}%" if not np.isnan(base_oos_sharpe) and 'oos_sharpe_delta' in variations else 'N/A',
                'Sensitivity': '\u26a0\ufe0f HIGH' if abs(variations['is_sharpe_delta'].min()) > 40 else '\u2705 LOW'
            })
        
        summary_df = pd.DataFrame(summary_data)
        print(summary_df.to_string(index=False))
        
        print("\n\u2705 Analysis Complete! Green = Robust, Red = Fragile")



In [ ]:
# ================================================================
# Universal Export Cell -- Professional PDF Tearsheet + Data Files
# ================================================================
# Uses shared lib/UNIVERSAL_EXPORT_CELL_v2.py for consistent output
# across all strategy notebooks.

STRATEGY_NAME = "RSI_MeanReversion"
PARAM_COLS = ['rsi_len', 'oversold', 'overbought']

import os
_lib_dir = 'lib' if os.path.isdir('lib') else os.path.join('..', 'lib')
exec(open(os.path.join(_lib_dir, 'UNIVERSAL_EXPORT_CELL_v2.py'), encoding='utf-8').read())
